[Documentation](https://superlinked.com/vectorhub/articles/optimizing-rag-with-hybrid-search-reranking)

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
import os

In [2]:
pdf_loader = PyPDFLoader("Data/rag_paper.pdf")
documents = pdf_loader.load()

In [3]:
# create chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 200,
    chunk_overlap = 30
)
chunks = splitter.split_documents(documents)

In [4]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs = {"device" : "cuda"}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
len(embedding_model.embed_query("Hello world"))

384

In [6]:
# create a vector store using the embeddings we obtain from the text.
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model
)

In [7]:
# build the keyword and semantic retrievers separately. 
# For keyword matching, we use the BM25 retriever from Langchain. 
# By setting k to 3, we’re asking the retriever to return the 3 most relevant 
# documents or vectors from the vector store.
vectorstore_retriver = vectorstore.as_retriever(
    search_kwargs = {'k' : 3}
)
keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever.k = 3

In [8]:
# now create an ensemble retriever, which is a weighted combination of the
# keyword retriver and vectorstore_retriver
ensemble_retriever = EnsembleRetriever(
    retrievers = [vectorstore_retriver , keyword_retriever],
    weights = [0.3 , 0.7]
)

In [9]:
query = "The RAG-Sequence model uses the same retrieved document to generate the complete sequence"

v_res = vectorstore_retriver.invoke(query)
k_res =keyword_retriever.invoke(query)
e_res = ensemble_retriever.invoke(query)

In [10]:
print(v_res[0].page_content)

byθ that generates a current token based on a context of the previousi− 1 tokensy1:i−1, the original
inputx and a retrieved passagez.
To train the retriever and generator end-to-end, we treat the retrieved document as a latent variable.
We propose two models that marginalize over the latent documents in different ways to produce a
distribution over generated text. In one approach, RAG-Sequence, the model uses the same document
to predict each target token. The second approach, RAG-Token, can predict each target token based
on a different document. In the following, we formally introduce both models and then describe the
pη andpθ components, as well as the training and decoding procedure.
2.1 Models
RAG-Sequence Model The RAG-Sequence model uses the same retrieved document to generate
the complete sequence. Technically, it treats the retrieved document as a single latent variable that
is marginalized to get the seq2seq probability p(y|x) via a top-K approximation. Concretely, the
top K 

In [11]:
print(k_res[0].page_content)

2.1 Models
RAG-Sequence Model The RAG-Sequence model uses the same retrieved document to generate
the complete sequence. Technically, it treats the retrieved document as a single latent variable that


In [12]:
print(e_res[0].page_content)

2.1 Models
RAG-Sequence Model The RAG-Sequence model uses the same retrieved document to generate
the complete sequence. Technically, it treats the retrieved document as a single latent variable that


In [13]:
# Load the model and tokenizer
model_name = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [14]:
# specify stop token ids
stop_token_ids = [0]

In [15]:
# from transformers import GenerationConfig
# gen_config = GenerationConfig(
#     # max_new_tokens=4096,
#     do_sample=True,
#     top_k=5,
#     num_return_sequences=1,
#     eos_token_id=tokenizer.eos_token_id,
#     pad_token_id=tokenizer.pad_token_id,
#     use_cache=True
# )

In [16]:
# model.generation_config = gen_config
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens = 4096,
    do_sample=True,
    top_k=5,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_k', 'do_sample', 'pad_token_id', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [17]:
from langchain_huggingface import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline = pipe)

In [18]:
from langchain_classic.prompts import PromptTemplate
prompt = PromptTemplate(
    template = """You are a assistant for question answering. Answer strictly from 
    the given context.
    
    Context:
    {context}
    
    Question:
    {input}
    
    Answer:
    """,
    input_variables = ["context" , "input"]
)

In [19]:
# create document chain
# to combine the retrieved documents into a single string to pass to the llm.
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

combine_docs_chain = create_stuff_documents_chain(
    llm = llm,
    prompt = prompt
)

In [20]:
# create the retrieval chain
# this will link the retriever and the combine_docs_chain
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(
    ensemble_retriever , combine_docs_chain
)

In [ ]:
response = retrieval_chain.invoke({
    "input": "Describe Generation Diversity."
})
response

Token indices sequence length is longer than the specified maximum sequence length for this model (4310 > 4096). Running this sequence through the model will result in indexing errors
Both `max_new_tokens` (=4096) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (4096). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
